<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/03_sessions_state.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 03 — Sessions, State, Events, Artifacts

Module 01 gave you the four primitives. Module 02 gave you tools. This module goes deep on **the third primitive — Session — and its contents: state, events, and artifacts.**

If Module 02 was about what an agent can *do*, Module 03 is about what an agent *remembers*. A stateless agent is a party trick. A stateful agent is infrastructure.

**What you'll build:**
- An agent that reads and writes session `state` from inside a tool.
- An agent that auto-saves its final response via `output_key=`.
- **The wow demo:** the same user starts two separate sessions — and the agent remembers their preferences in session two because of the `user:` state prefix.

**Running cost:** under $0.01 on OpenRouter.

# Setup

## Install Dependencies

In [1]:
!pip install -q google-adk==2.4.0 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


## API Key Configuration

In [2]:
import os

OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    print("💡 Set OPENROUTER_API_KEY in Colab secrets (🔑 icon) or a local .env file.")
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

MODEL_STRING = "openrouter/google/gemini-2.5-flash-lite"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/google/gemini-2.5-flash-lite


## Import Libraries

In [3]:
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")
try:
    sys.stderr.fileno()
except Exception:
    sys.stderr = open(os.devnull, "w")

import nest_asyncio; nest_asyncio.apply()
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.tool_context import ToolContext
from google.genai import types

print("✅ Imports successful.")

09:17:59 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


09:17:59 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


✅ Imports successful.


# Sessions — A Refresher

A `Session` is identified by a triple: `(app_name, user_id, session_id)`. It holds two things:

- **A list of events** — the full ordered history of every message, tool call, tool response, and state mutation. Read-only from your code's perspective; ADK appends to it.
- **A state dict** — mutable key-value storage for anything you want the agent to carry between turns.

Three session services ship with ADK:

| Service | Backing | When |
|---|---|---|
| `InMemorySessionService` | Python dict | Demos and tests |
| `DatabaseSessionService` | SQLAlchemy (Postgres, SQLite) | Self-hosted production |
| `VertexAiSessionService` | Google Cloud | Vertex deployments |

We'll use in-memory through M07. When memory becomes the point in M08, we'll swap to the database version.

# State — With Scope Prefixes

The most under-documented feature of ADK, and the one you'll use every day.

A state dict is just a Python dict, but **the key's prefix decides where it lives and how long it survives.**

| Prefix | Scope | Survives |
|---|---|---|
| *(none)* | This session only | Until the session is deleted |
| `user:` | This user, across all their sessions | As long as the user exists |
| `app:` | Global across all users of this app | As long as the app exists |
| `temp:` | This invocation only | Thrown away after the current run |

Four scope rings. Think of it as React state with persistence tiers baked in — local, user profile, global config, throwaway scratch.

The demos below make this concrete.

# Demo 1 — Writing State From a Tool

The cleanest way to mutate state is from inside a tool. Tools that take a `tool_context: ToolContext` argument get handed the running context, and `tool_context.state` behaves like a dict you can assign to. ADK records the mutation as an event, and by the next turn the new state is visible to the agent.

In [4]:
APP = "m03_demo"
USER = "alice"
session_service = InMemorySessionService()

def remember_favorite_color(color: str, tool_context: ToolContext) -> dict:
    """Record the user's favorite color for future sessions.

    Args:
        color: A color name like "teal", "crimson", "forest green".
    """
    # Note the user: prefix — this makes the value survive beyond this session.
    tool_context.state["user:favorite_color"] = color
    return {"status": "saved", "color": color}


def recall_favorite_color(tool_context: ToolContext) -> dict:
    """Check whether the user has told us their favorite color before."""
    color = tool_context.state.get("user:favorite_color")
    if color:
        return {"known": True, "color": color}
    return {"known": False}


color_agent = LlmAgent(
    name="color_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Remembers and recalls the user's favorite color across sessions.",
    instruction=(
        "You track the user's favorite color. "
        "When they tell you a color, call remember_favorite_color. "
        "When they ask what their color is, call recall_favorite_color first. "
        "Be brief."
    ),
    tools=[remember_favorite_color, recall_favorite_color],
    output_key="last_response",   # also save the final text to state['last_response']
)

print("✅ color_agent ready.")

✅ color_agent ready.


## Run Session 1 — tell the agent your color

In [5]:
async def chat(agent, prompt: str, sid: str):
    sess = await session_service.get_session(app_name=APP, user_id=USER, session_id=sid)
    if sess is None:
        await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER ({sid}): {prompt}")
    async for event in runner.run_async(user_id=USER, session_id=sid, new_message=message):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text and p.text.strip():
                    tag = "[FINAL]" if event.is_final_response() else "[step]"
                    print(f"{tag} {event.author}: {p.text.strip()[:200]}")
                if p.function_call:
                    print(f"[tool_call] {p.function_call.name}({dict(p.function_call.args)})")
                if p.function_response:
                    print(f"[tool_resp] {p.function_response.response}")
    print()

SID_1 = "session-one"
await chat(color_agent, "My favorite color is teal.", SID_1)

# Inspect what's in the session's state now.
s1 = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_1)
print("── Session 1 state after the conversation ──")
for k, v in sorted(dict(s1.state).items()):
    print(f"  {k!r}: {v!r}")

USER (session-one): My favorite color is teal.


[tool_call] remember_favorite_color({'color': 'teal'})
[tool_resp] {'status': 'saved', 'color': 'teal'}


[FINAL] color_agent: You got it! I'll remember that your favorite color is teal.

── Session 1 state after the conversation ──
  'last_response': "You got it! I'll remember that your favorite color is teal."
  'user:favorite_color': 'teal'


Two things happened. The tool wrote `user:favorite_color = "teal"` into the session's state. Separately, `output_key="last_response"` on the agent auto-saved the model's final text reply under the key `last_response`. Both are now part of the session. Both showed up in the event stream if you scroll up.

Next: we'll open a **completely separate session** for the same user. The `user:`-prefixed key should carry over. The `last_response` key (unprefixed) should not.

## Run Session 2 — fresh session, same user

In [6]:
SID_2 = "session-two"

# Create the second session fresh — notice we don't pass any initial state.
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID_2)
s2_initial = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)

print("── Session 2 initial state (before any chat) ──")
for k, v in sorted(dict(s2_initial.state).items()):
    print(f"  {k!r}: {v!r}")
print()

# Ask the agent in session 2 — does it remember?
await chat(color_agent, "Hey, what's my favorite color?", SID_2)

── Session 2 initial state (before any chat) ──
  'user:favorite_color': 'teal'

USER (session-two): Hey, what's my favorite color?


[tool_call] recall_favorite_color({})
[tool_resp] {'known': True, 'color': 'teal'}


[FINAL] color_agent: Teal.



Read the output carefully.

- Session 2 **starts** with `user:favorite_color: 'teal'` already present — it survived.
- `last_response` from Session 1 is **not** present — it was unprefixed, so it stayed behind.
- The agent called `recall_favorite_color`, which read `user:favorite_color` and returned `"teal"` — which the model then reported back.

This is cross-session memory in 80 lines of code. No database, no vector store, no custom embedding pipeline — just a prefix convention on a dict key.

**What the scope prefixes give you, in one sentence:** you decide the lifetime of every piece of data with a 5-character prefix, and ADK handles the rest.

# Three Ways to Write State — And One That Doesn't Persist

You just saw two of them. Here's the complete picture:

| Pattern | When it runs | Persists? | Use for |
|---|---|---|---|
| `output_key="last_response"` on the agent | After the model produces final text | **Yes** | Caching the last reply, passing output between workflow steps |
| `tool_context.state[key] = value` inside a tool | When the tool executes | **Yes** | Writing structured data during tool calls |
| `session.state[key] = value` directly on a fetched session | As soon as you assign | **No — it does not persist!** | Avoid |

The third one is the pitfall: if you fetch a session with `get_session()` and mutate its `.state` directly, the mutation **does not persist to the session service**. ADK persists state via events — so mutations from `output_key` and from `tool_context.state` get recorded as events. Direct dict assignment bypasses that, and the next `get_session()` returns the old state.

**Rule:** use `output_key=` on the agent, or `tool_context.state[...]` inside a tool. Never assign to a returned session's `.state` and expect it to stick.

In [7]:
# Demonstrate the pitfall — do NOT copy this pattern.
# Mutate state directly and re-fetch to prove it didn't stick.
sess = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)
sess.state["user:mood"] = "happy"                 # direct assignment — WRONG
sess.state["this_does_not_persist"] = "ghost"     # unprefixed — also wrong, but doubly so

sess_fresh = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)
print("── Re-fetched session state ──")
for k, v in sorted(dict(sess_fresh.state).items()):
    print(f"  {k!r}: {v!r}")
print()
print("Notice: 'user:mood' and 'this_does_not_persist' are NOT in the re-fetched state.")
print("Direct assignment to a returned session's .state does not round-trip.")

── Re-fetched session state ──
  'last_response': 'Teal.'
  'user:favorite_color': 'teal'

Notice: 'user:mood' and 'this_does_not_persist' are NOT in the re-fetched state.
Direct assignment to a returned session's .state does not round-trip.


# Events — The Session's Immutable Ledger

Every turn in a session produces events, and the session keeps them all. You can walk the event history to audit what happened, replay a conversation, or feed it into an evaluation suite.

The one-line version: **a session's event list is the agent's log file, with structure.**

In [8]:
# Show the event history of session 1.
s1 = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_1)
print(f"Session 1 has {len(s1.events)} events.\n")
for i, ev in enumerate(s1.events):
    author = ev.author or "(system)"
    kinds = []
    if ev.content and ev.content.parts:
        for p in ev.content.parts:
            if p.text and p.text.strip():
                kinds.append(f"text({p.text.strip()[:40]}...)" if len(p.text.strip()) > 40 else f"text({p.text.strip()})")
            if p.function_call:
                kinds.append(f"tool_call({p.function_call.name})")
            if p.function_response:
                kinds.append(f"tool_resp({p.function_response.name if hasattr(p.function_response, 'name') else '?'})")
    if ev.actions and ev.actions.state_delta:
        kinds.append(f"state_delta({list(ev.actions.state_delta.keys())})")
    print(f"  [{i:>2}] {author:16s} {' | '.join(kinds) if kinds else '(no content)'}")

Session 1 has 4 events.

  [ 0] user             text(My favorite color is teal.)
  [ 1] color_agent      tool_call(remember_favorite_color)
  [ 2] color_agent      tool_resp(remember_favorite_color) | state_delta(['user:favorite_color'])
  [ 3] color_agent      text(You got it! I'll remember that your favo...) | state_delta(['last_response'])


Notice the `state_delta` entries in the event stream. Every time state was written — by the tool, by `output_key=`, by session initialization — a corresponding event was recorded. The state dict is a *projection* of this event stream. ADK replays events when you fetch a session; the state dict is what you get when those deltas are applied in order.

This is [event sourcing](https://martinfowler.com/eaaDev/EventSourcing.html). Skimmable today; it'll come back as a design principle in M08 when we replace the in-memory session service with a database.

# Artifacts — Binary Blobs Outside the Event Stream

A quick word on the fourth concept and then we wrap.

**Artifacts** are for binary data — images, audio clips, PDFs, any large blob — that you want associated with a session but don't want to serialize into events. The analogy is [Git LFS](https://git-lfs.com) for agents: the pointer lives in the event stream; the payload lives in a separate store.

Three built-in artifact services mirror the session services:

- `InMemoryArtifactService` — demos and tests.
- `GcsArtifactService` — Google Cloud Storage, for production.
- (and you can implement your own `BaseArtifactService` for S3, Azure Blob, MinIO, etc.)

You'll typically only need artifacts for multi-modal agents — one that accepts an uploaded image, or one whose tool generates a PDF report. For the text-only agents in Part 1 of this course, you can ignore artifacts entirely. M13 (Live API voice agent) is the first module where artifacts carry real weight.

# Interlude — Memory Staleness and Verification

> *From "Agentic Design Patterns," Chapter 2. Two minutes of theory that pays off for the rest of the course.*

The wow demo in this module looks clean because the state has not had time to go stale. In production, that is not how this works.

A user's favorite color from three months ago is probably still their favorite color. Their current project? Maybe not — they might have switched to a new one and never told the agent. The tickets the agent "remembered" being open yesterday might all be closed by now. The server IP an MCP tool cached might have been reassigned.

**Stored memory is a hint, not a fact.**

Three guidelines, from the publication:

1. **Prefer retrieval over recall for high-stakes decisions.** If the agent is about to do something that depends on state being accurate (send an email, charge a card, deploy code), it should re-verify the state by calling a read-only tool rather than trusting its own stored state.
2. **Scope your state aggressively.** `temp:` for throwaway scratchpads. Unprefixed for per-session. `user:` for things you're confident change only with explicit user action. Reserve `app:` for genuinely immutable configuration.
3. **Log a reason when you write to `user:` or `app:` state.** Months from now you'll debug an agent that's acting on six-month-old memory; a one-line "why did this get stored" note saves hours of investigation.

The pattern has a memorable name in the publication: **Skeptical Memory.** Treat your own stored context as unverified until proven otherwise. M08 picks up this thread when we move past session state into long-term memory.

# Your Turn

Four small changes. Run in new cells below.

1. **App-scope state.** Write a tool `set_app_mode(mode: str, tool_context)` that stores `tool_context.state["app:mode"] = mode`. Create two separate users (change `USER` between calls). Does `app:mode` survive across users?
2. **Temp state.** Write a tool that stores `tool_context.state["temp:scratch"] = "..."`. After the run completes, fetch the session again and inspect the state. Is `temp:scratch` still there?
3. **Replay a conversation.** Write a loop that walks `sess.events` and prints only the user's and agent's text messages in order — a clean transcript.
4. **Intentional staleness.** Make the agent save a favorite color, then manually change `session.state["user:favorite_color"]` to something else via a direct (non-persisting) assignment. Ask the agent what the color is. Which value does it see — the persisted one, or your uncommitted change?

# Key Takeaways

- A **Session** is a triple `(app_name, user_id, session_id)` that holds events and a state dict.
- **State prefixes** are the most useful under-documented feature: `user:` survives the session, `app:` is global, `temp:` is per-invocation, unprefixed is per-session.
- **Two ways to write state that actually persist:** `output_key=` on the agent, and `tool_context.state[...]` inside a tool. Direct assignment on a returned session's `.state` does NOT round-trip.
- **Events are the session's immutable ledger.** State is a projection of the event stream's state-delta entries.
- **Artifacts** hold binary data outside the event stream. Think Git LFS for agents.
- **Interlude — Skeptical Memory:** stored state is a hint, not a fact. Verify before acting on high-stakes decisions.

# Next up — M04: The one-line model swap

You've been using `LiteLlm(model="openrouter/google/gemini-2.5-flash-lite")` the whole time. M04 opens that abstraction up: how LiteLLM actually works, how to route to Claude, GPT, Qwen, and a locally-hosted Ollama model, and the specific gotcha around the `ollama_chat/` prefix that causes infinite tool-call loops if you get it wrong.